In [1]:
from pathlib import Path

from PIL import Image
import pandas as pd
import pymupdf
import pytesseract

REPO_ROOT = Path.cwd() if (Path.cwd() / "KB Articles.pdf").exists() else Path.cwd().parent
kb_pdf = REPO_ROOT / "KB Articles.pdf"
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MIN_EXTRACTED_CHARS = 50
OCR_ZOOM = 2
KB_DB_BASE_PATH = REPO_ROOT / "kb_db"

kb_pdf, KB_DB_BASE_PATH

(PosixPath('/Users/nbilla/Documents/GitHub/nivibilla/lng-ticket-assistant/KB Articles.pdf'),
 PosixPath('/Users/nbilla/Documents/GitHub/nivibilla/lng-ticket-assistant/kb_db'))

In [2]:
def extract_page_text(page: pymupdf.Page) -> str:
    text = page.get_text("text").strip()
    if len(text) >= MIN_EXTRACTED_CHARS:
        return text
    pix = page.get_pixmap(matrix=pymupdf.Matrix(OCR_ZOOM, OCR_ZOOM))
    image = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    return pytesseract.image_to_string(image).strip()


doc = pymupdf.open(kb_pdf)
pages = []
for page_number, page in enumerate(doc, start=1):
    pages.append({"page": page_number, "text": extract_page_text(page)})
doc.close()

pages_df = pd.DataFrame(pages)
pages_df.assign(n_chars=pages_df["text"].str.len())

,page,text,n_chars
0,1,Synthetic Knowledge Base Articles — Incident\n...,2195
1,2,Escalation trigger: User does not have MFA con...,2186
2,3,Confirm your network password hasn't expired v...,2296
3,4,Search for the application by name.\nSubmit th...,2062
4,5,"In Outlook, go to Send/Receive > Update Folder...",2107
5,6,Try printing a test page directly from the pri...,1905
6,7,Confirm system time on your phone is set to au...,2083
7,8,Related Articles: KB0020002 (VPN connects but ...,2267
8,9,"KB0080002 — Teams call quality poor, freezing,...",1086


In [3]:
def recursive_split(
    text: str,
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
    separators: tuple[str, ...] = ("\n\n", "\n", ". ", " "),
) -> list[str]:
    if not text:
        return []
    if len(text) <= chunk_size:
        return [text]

    separator = separators[0] if separators else ""
    remaining = separators[1:] if separators else ()
    parts = text.split(separator) if separator else list(text)

    chunks: list[str] = []
    current: list[str] = []
    current_len = 0

    def current_text() -> str:
        return separator.join(current) if separator else "".join(current)

    def overflow(piece: str) -> bool:
        extra = len(separator) if current and separator else 0
        return current_len + extra + len(piece) > chunk_size

    for part in parts:
        if overflow(part) and current:
            merged = current_text()
            if len(merged) > chunk_size and remaining:
                chunks.extend(recursive_split(merged, chunk_size, chunk_overlap, remaining))
            else:
                chunks.append(merged)
            while current and current_len > chunk_overlap:
                dropped = current.pop(0)
                current_len -= len(dropped)
                if current:
                    current_len -= len(separator)
        if overflow(part) and not current:
            if remaining:
                chunks.extend(recursive_split(part, chunk_size, chunk_overlap, remaining))
            else:
                start = 0
                while start < len(part):
                    end = min(start + chunk_size, len(part))
                    chunks.append(part[start:end])
                    if end == len(part):
                        break
                    start = max(end - chunk_overlap, start + 1)
            continue
        if current:
            current_len += len(separator) + len(part)
        else:
            current_len = len(part)
        current.append(part)

    if current:
        merged = current_text()
        if len(merged) > chunk_size and remaining:
            chunks.extend(recursive_split(merged, chunk_size, chunk_overlap, remaining))
        else:
            chunks.append(merged)
    return [chunk for chunk in chunks if chunk.strip()]


def pages_for_span(spans: list[tuple[int, int, int]], start: int, end: int) -> list[int]:
    return [page for page, span_start, span_end in spans if span_start < end and span_end > start]


parts: list[str] = []
page_spans: list[tuple[int, int, int]] = []
cursor = 0
for i, row in enumerate(pages):
    if i:
        parts.append("\n\n")
        cursor += 2
    start = cursor
    parts.append(row["text"])
    cursor += len(row["text"])
    page_spans.append((row["page"], start, cursor))

full_text = "".join(parts)
raw_chunks = recursive_split(full_text)

records = []
search_from = 0
for chunk_id, text in enumerate(raw_chunks):
    start = full_text.find(text, search_from)
    if start < 0:
        start = full_text.find(text)
    end = start + len(text)
    covered = pages_for_span(page_spans, start, end)
    records.append(
        {
            "chunk_id": chunk_id,
            "source": kb_pdf.name,
            "page_start": min(covered) if covered else None,
            "page_end": max(covered) if covered else None,
            "text": text,
            "n_chars": len(text),
        }
    )
    search_from = max(start + 1, end - CHUNK_OVERLAP)

chunks_df = pd.DataFrame.from_records(records)
chunks_df

,chunk_id,source,page_start,page_end,text,n_chars
0,0,KB Articles.pdf,1,1,Synthetic Knowledge Base Articles — Incident\n...,907
1,1,KB Articles.pdf,1,1,"Cause — brief root cause, gives the article mo...",805
2,2,KB Articles.pdf,1,2,Symptoms: User reports being unable to log in ...,941
3,3,KB Articles.pdf,2,2,Escalation trigger: User does not have MFA con...,977
4,4,KB Articles.pdf,2,2,Wait for the 30-minute auto-unlock rather than...,831
5,5,KB Articles.pdf,2,3,Symptoms: VPN client shows a connection failur...,991
6,6,KB Articles.pdf,3,3,Escalation trigger: VPN still fails after a co...,965
7,7,KB Articles.pdf,3,3,Try accessing a different internal system to e...,976
8,8,KB Articles.pdf,3,4,Symptoms: User cannot open or is not licensed ...,872
9,9,KB Articles.pdf,4,4,Related Articles: KB0030002 (Access Denied err...,886


In [4]:
print(f"{len(chunks_df)} chunks from {kb_pdf.name}")
display(chunks_df.head())
print("\n--- chunk 0 ---\n")
print(chunks_df.loc[0, "text"])
if len(chunks_df) > 1:
    print("\n--- last chunk ---\n")
    print(chunks_df.iloc[-1]["text"])

23 chunks from KB Articles.pdf


,chunk_id,source,page_start,page_end,text,n_chars
0,0,KB Articles.pdf,1,1,Synthetic Knowledge Base Articles — Incident\n...,907
1,1,KB Articles.pdf,1,1,"Cause — brief root cause, gives the article mo...",805
2,2,KB Articles.pdf,1,2,Symptoms: User reports being unable to log in ...,941
3,3,KB Articles.pdf,2,2,Escalation trigger: User does not have MFA con...,977
4,4,KB Articles.pdf,2,2,Wait for the 30-minute auto-unlock rather than...,831



--- chunk 0 ---

Synthetic Knowledge Base Articles — Incident
Deflection Prototype

Purpose: Synthetic KB content for testing ServiceNow Virtual Agent Knowledge Search deflection logic.
Not real L&G documentation — written for this prototype only, per the project's GDPR/data-handling
approach (no live employee or company data used in testing).

Scope: P3 (Moderate) / P4 (Low) priority incidents only — low-complexity, high-volume issue types
suitable for self-service resolution. P1/P2 incidents are explicitly out of scope for deflection and should
always route straight to a ticket.

Template fields used (matches standard ServiceNow KB structure):

Symptoms — what the user actually sees/reports (this is closest to real user phrasing, useful for NLU
training)

Cause — brief root cause, gives the article more matchable text and makes it feel like genuine
documentation, not just a script

Resolution — numbered steps

--- last chunk ---

Escalation trigger: Poor quality persists consistentl

In [5]:
import lancedb
import pyarrow as pa

kb_db = lancedb.connect(KB_DB_BASE_PATH)

knowledge_base_schema = pa.schema(
    [
        pa.field("chunk_id", pa.int64()),
        pa.field("source", pa.string()),
        pa.field("page_start", pa.int64()),
        pa.field("page_end", pa.int64()),
        pa.field("text", pa.string()),
        pa.field("n_chars", pa.int64()),
    ]
)

try:
    kb_db.drop_table("knowledge_base")
except ValueError:
    pass

knowledge_base_table = kb_db.create_table("knowledge_base", data=chunks_df, mode="overwrite")

[2026-09-05T16:26:26Z WARN  lance::dataset::write::insert] No existing dataset at /Users/nbilla/Documents/GitHub/nivibilla/lng-ticket-assistant/kb_db/knowledge_base.lance, it will be created
